
# FleetLogix — Primer avance

## 1. Análisis del modelo de datos

La base de datos FleetLogix representa las operaciones de una empresa de transporte y logística. Está formada por seis tablas relacionadas: vehicles, drivers, routes, trips, deliveries y maintenance.

Las tablas vehicles, drivers y routes almacenan los datos maestros. Las tablas trips, deliveries y maintenance registran los viajes, las entregas y los mantenimientos de los vehículos.

### 1.1. Justificación de las tablas

- **vehicles:** almacena los vehículos, su capacidad de carga, combustible, fecha de adquisición y estado. Permite identificar qué vehículo realiza cada viaje y recibe cada mantenimiento.

- **drivers:** almacena los conductores, sus datos de identificación, licencia, fecha de contratación y estado. Permite asociar cada viaje con su conductor.

- **routes:** almacena las rutas entre ciudades, sus distancias, duraciones estimadas y costos de peajes. Permite reutilizar esta información en distintos viajes.

- **trips:** registra cada viaje y lo relaciona con un vehículo, un conductor y una ruta. Incluye fechas de salida y llegada, combustible consumido, peso transportado y estado.

- **deliveries:** registra los paquetes y sus entregas, incluyendo número de seguimiento, cliente, dirección, peso, fechas y estado. Permite asociar varias entregas con un mismo viaje.

- **maintenance:** registra los mantenimientos de cada vehículo, su tipo, fecha, costo, descripción, responsable y próxima fecha de mantenimiento. Permite conservar el historial de servicios de la flota.

### 1.2. Relaciones entre las tablas

| Tabla principal | Tabla dependiente | Clave foránea | Relación |
|---|---|---|---|
| vehicles | trips | trips.vehicle_id → vehicles.vehicle_id | Un vehículo puede realizar muchos viajes. |
| drivers | trips | trips.driver_id → drivers.driver_id | Un conductor puede realizar muchos viajes. |
| routes | trips | trips.route_id → routes.route_id | Una ruta puede utilizarse en muchos viajes. |
| trips | deliveries | deliveries.trip_id → trips.trip_id | Un viaje puede incluir muchas entregas. |
| vehicles | maintenance | maintenance.vehicle_id → vehicles.vehicle_id | Un vehículo puede tener muchos mantenimientos. |

Las cinco relaciones son de uno a muchos (1:N). Las claves foráneas impiden guardar un identificador no nulo que no exista en la tabla principal.

En el esquema actual, estas columnas permiten valores NULL: la clave foránea por sí sola no obliga a asignar la relación. En los datos cargados se comprobó que todos los registros dependientes tienen una referencia válida.

### 1.3. Restricciones del modelo (constraints)

#### Claves primarias — PRIMARY KEY

Cada tabla tiene un identificador único que no permite valores NULL ni duplicados:

| Tabla | Clave primaria |
|---|---|
| vehicles | vehicle_id |
| drivers | driver_id |
| routes | route_id |
| trips | trip_id |
| deliveries | delivery_id |
| maintenance | maintenance_id |

#### Valores únicos — UNIQUE

Los siguientes campos no permiten valores duplicados:

- vehicles: license_plate.
- drivers: employee_code y license_number.
- routes: route_code.
- deliveries: tracking_number.

#### Campos obligatorios — NOT NULL

Además de las claves primarias, los siguientes campos no permiten valores NULL:

| Tabla | Campos obligatorios |
|---|---|
| vehicles | license_plate, vehicle_type |
| drivers | employee_code, first_name, last_name, license_number |
| routes | route_code, origin_city, destination_city |
| trips | departure_datetime |
| deliveries | tracking_number, customer_name, delivery_address |
| maintenance | maintenance_date |

#### Claves foráneas — FOREIGN KEY

El modelo contiene las cinco claves foráneas descritas en la sección 1.2. Estas comprueban que cada identificador no nulo exista en su tabla de referencia.

#### Valores predeterminados — DEFAULT

- vehicles.status y drivers.status: 'active'.
- trips.status: 'in_progress'.
- deliveries.delivery_status: 'pending'.
- deliveries.recipient_signature: FALSE.
- routes.toll_cost: 0.

Los identificadores definidos como SERIAL reciben automáticamente un valor de una secuencia.

#### Reglas comprobadas mediante validaciones

El esquema no define restricciones CHECK. Por ello, condiciones como llegada posterior a salida y peso de los paquetes dentro de la carga del viaje se comprobaron mediante consultas SQL; no son restricciones automáticas de la base.

### 1.3. Restricciones del modelo (constraints)

#### Claves primarias — PRIMARY KEY

Cada tabla tiene una clave primaria que identifica de forma única sus registros. No permite valores duplicados ni valores NULL.

| Tabla | Clave primaria |
|---|---|
| vehicles | vehicle_id |
| drivers | driver_id |
| routes | route_id |
| trips | trip_id |
| deliveries | delivery_id |
| maintenance | maintenance_id |

#### Valores únicos — UNIQUE

Estas restricciones impiden que se repitan los valores de los siguientes campos:

| Tabla | Campo con restricción UNIQUE |
|---|---|
| vehicles | license_plate |
| drivers | employee_code |
| drivers | license_number |
| routes | route_code |
| deliveries | tracking_number |

En drivers, employee_code y license_number son únicos de forma independiente: no se pueden repetir los códigos de empleado ni los números de licencia.

#### Campos obligatorios — NOT NULL

Además de las claves primarias, los siguientes campos no permiten valores NULL:

| Tabla | Campos obligatorios |
|---|---|
| vehicles | license_plate, vehicle_type |
| drivers | employee_code, first_name, last_name, license_number |
| routes | route_code, origin_city, destination_city |
| trips | departure_datetime |
| deliveries | tracking_number, customer_name, delivery_address |
| maintenance | maintenance_date |

NOT NULL exige que exista un valor, pero por sí solo no impide guardar una cadena de texto vacía.

#### Claves foráneas — FOREIGN KEY

El modelo contiene cinco claves foráneas:

- trips.vehicle_id → vehicles.vehicle_id.
- trips.driver_id → drivers.driver_id.
- trips.route_id → routes.route_id.
- deliveries.trip_id → trips.trip_id.
- maintenance.vehicle_id → vehicles.vehicle_id.

Estas restricciones exigen que cada identificador no nulo exista en la tabla referenciada.

Las cinco columnas permiten NULL en el esquema actual. Por eso, la clave foránea comprueba la existencia de la referencia cuando se proporciona un identificador, pero no obliga a proporcionarlo.

#### Valores predeterminados — DEFAULT

| Tabla | Campo | Valor predeterminado |
|---|---|---|
| vehicles | status | 'active' |
| drivers | status | 'active' |
| trips | status | 'in_progress' |
| deliveries | delivery_status | 'pending' |
| deliveries | recipient_signature | FALSE |
| routes | toll_cost | 0 |

Estos valores se utilizan cuando el campo se omite en un INSERT o se indica DEFAULT. No sustituyen un NULL enviado explícitamente.

Las claves primarias definidas como SERIAL reciben automáticamente un identificador generado mediante una secuencia cuando se omite su valor.

#### Reglas de negocio y restricciones CHECK

El esquema proporcionado no define restricciones CHECK.

Por ello, las claves primarias, claves foráneas, restricciones UNIQUE y NOT NULL no garantizan por sí solas todas las reglas de negocio.

Mediante consultas SQL se comprobó que:

- No hubiera viajes con llegada anterior o igual a la salida.
- Las entregas realizadas estuvieran dentro del intervalo de su viaje.
- La suma del peso de los paquetes no superara el peso registrado para el viaje.

Las consultas ejecutadas devolvieron cero casos que incumplieran estas condiciones. Sus resultados se documentarán en la sección de control de calidad.

### 1.4. Diagrama entidad-relación

El siguiente diagrama muestra las seis tablas del modelo y sus cinco relaciones de uno a muchos (1:N).

![Diagrama entidad-relación de FleetLogix](FleetLogix_Diagrama_ER.png)

## 2. Generación de datos sintéticos

Se utilizó el script A1-01_data_generation_estudiantes.py para generar y cargar datos sintéticos en PostgreSQL.

La carga se realizó respetando las dependencias entre tablas: primero vehicles, drivers y routes; después trips; finalmente deliveries y maintenance.

### 2.1. Cantidad de registros cargados

Los siguientes conteos se comprobaron mediante consultas SQL en DBeaver:

| Tabla | Tipo | Registros |
|---|---|---:|
| vehicles | Maestra | 200 |
| drivers | Maestra | 400 |
| routes | Maestra | 50 |
| trips | Transaccional | 100.000 |
| deliveries | Transaccional | 400.000 |
| maintenance | Transaccional | 5.000 |
| **Total** | | **505.650** |

### 2.2. Funcionamiento de generate_trips()

El método generate_trips(count=100000) genera y carga los viajes en PostgreSQL.

1. Consulta los vehículos y conductores activos y las rutas existentes. Utiliza sus identificadores para establecer las relaciones de cada viaje.

2. Establece una fecha inicial de 730 días antes del momento de ejecución.

3. Para cada viaje, selecciona aleatoriamente un vehículo, un conductor y una ruta.

4. Selecciona la hora de salida mediante las probabilidades del auxiliar _get_hourly_distribution() y asigna un minuto aleatorio.

5. Calcula la duración multiplicando el tiempo estimado de la ruta por un factor aleatorio entre 0,8 y 1,3. La llegada se obtiene sumando esa duración a la salida.

6. Estima el combustible multiplicando la distancia por un consumo entre 0,08 y 0,15 litros por kilómetro. Asigna una carga entre el 40 % y el 90 % de la capacidad del vehículo.

7. Si la llegada calculada ya pasó, asigna el estado completed y guarda esa fecha. En caso contrario, asigna in_progress y deja arrival_datetime como NULL.

8. Almacena los viajes en una lista y los inserta mediante execute_batch en grupos de 1.000 registros. Confirma cada grupo con commit() y registra el progreso mediante logging.

**Observación de la versión revisada:** el avance de fecha utiliza int(), que elimina los decimales del intervalo. Para 100.000 viajes, esto reduce el período generado a aproximadamente 694 días. Este cálculo requiere ajuste para cubrir los 730 días previstos.

### 2.3. Funcionamiento de _get_hourly_distribution()

Este método auxiliar devuelve una probabilidad para cada una de las 24 horas del día.

Primero asigna los siguientes pesos:

| Horas | Peso por hora |
|---|---:|
| 00:00–05:59 y 20:00–23:59 | 0,02 |
| 06:00–07:59, 12:00–13:59 y 18:00–19:59 | 0,06 |
| 08:00–11:59 | 0,08 |
| 14:00–17:59 | 0,07 |

Después divide cada peso entre la suma de todos los pesos. Esto convierte los valores en probabilidades cuya suma es 1.

generate_trips() utiliza estas probabilidades mediante np.random.choice() para seleccionar la hora de salida.

Así, las salidas son más probables durante el horario diurno, especialmente entre las 08:00 y las 11:59. Los pesos iniciales no representan porcentajes finales.

## 3. Control de calidad de los datos

Después de la carga, se ejecutaron consultas SQL para comprobar las cantidades y la coherencia de los registros.

### 3.1. Validaciones realizadas

| Validación | Casos que incumplen |
|---|---:|
| Viajes sin vehículo, conductor o ruta existente | 0 |
| Mantenimientos sin vehículo existente | 0 |
| Viajes con llegada anterior o igual a la salida | 0 |
| Entregas realizadas fuera del intervalo de su viaje | 0 |
| Viajes cuya suma de pesos de paquetes supera el peso registrado del viaje | 0 |
| Viajes con menos de 2 o más de 6 entregas | 0 |

En estas consultas, cero significa que no se encontraron registros que incumplieran la condición evaluada. No significa que las tablas estén vacías.

### 3.2. Distribución de entregas por viaje

Mediante una consulta SQL se contó cuántas entregas tiene cada viaje y cuántos viajes corresponden a cada cantidad.

| Entregas por viaje | Cantidad de viajes |
|---|---:|
| 2 | 10.000 |
| 3 | 20.000 |
| 4 | 40.000 |
| 5 | 20.000 |
| 6 | 10.000 |
| **Total** | **100.000** |

Los resultados corresponden a 400.000 entregas. Todos los viajes tienen entre 2 y 6 entregas, y el 40 % tiene 4 entregas, siendo la cantidad más frecuente.

### 3.3. Registro de la carga y reporte de resultados

El script utiliza logging para registrar mensajes con fecha, hora y nivel de importancia, tanto en la terminal como en el archivo data_generation.log.

Estos mensajes permiten revisar:

- La conexión con PostgreSQL.
- El inicio y el progreso de la carga.
- Las cantidades de registros insertados.
- Los errores que ocurren durante la ejecución.
- Los resultados de las validaciones implementadas en el script.
- El cierre de la conexión.

También se genera el archivo generation_summary.json como reporte de la ejecución.

Los conteos finales de las seis tablas se comprobaron de forma independiente mediante consultas SQL en DBeaver.

### 3.4. Consultas SQL de validación

#### Integridad referencial de los viajes

Se utilizó LEFT JOIN para buscar viajes sin una referencia válida a vehículos, conductores o rutas.

```sql
SELECT COUNT(*) AS viajes_sin_relacion_valida
FROM trips t
LEFT JOIN vehicles v ON t.vehicle_id = v.vehicle_id
LEFT JOIN drivers d ON t.driver_id = d.driver_id
LEFT JOIN routes r ON t.route_id = r.route_id
WHERE v.vehicle_id IS NULL
   OR d.driver_id IS NULL
   OR r.route_id IS NULL;
```

**Resultado obtenido en DBeaver:** 0 viajes sin relación válida.

### 3.4. Consultas SQL de validación

#### Integridad referencial de los viajes

Se utilizó LEFT JOIN para buscar viajes sin una referencia válida a vehículos, conductores o rutas.

```sql
SELECT COUNT(*) AS viajes_sin_relacion_valida
FROM trips t
LEFT JOIN vehicles v ON t.vehicle_id = v.vehicle_id
LEFT JOIN drivers d ON t.driver_id = d.driver_id
LEFT JOIN routes r ON t.route_id = r.route_id
WHERE v.vehicle_id IS NULL
   OR d.driver_id IS NULL
   OR r.route_id IS NULL;
```

**Resultado obtenido en DBeaver:** 0 viajes sin relación válida.
#### Consistencia temporal de los viajes

Se buscaron viajes cuya llegada fuera anterior o igual a la salida.

```sql
SELECT COUNT(*) AS viajes_con_fechas_invalidas
FROM trips
WHERE arrival_datetime <= departure_datetime;
```

**Resultado obtenido en DBeaver:** 0 viajes con llegada anterior o igual a la salida.

Esta consulta evalúa los viajes con fecha de llegada registrada. Los valores NULL de arrival_datetime no se incluyen en la comparación.

### 3.4. Consultas SQL de validación

#### Integridad referencial de los viajes

Se utilizó LEFT JOIN para buscar viajes sin una referencia válida a vehículos, conductores o rutas.

```sql
SELECT COUNT(*) AS viajes_sin_relacion_valida
FROM trips t
LEFT JOIN vehicles v ON t.vehicle_id = v.vehicle_id
LEFT JOIN drivers d ON t.driver_id = d.driver_id
LEFT JOIN routes r ON t.route_id = r.route_id
WHERE v.vehicle_id IS NULL
   OR d.driver_id IS NULL
   OR r.route_id IS NULL;
```

**Resultado obtenido en DBeaver:** 0 viajes sin relación válida.
#### Consistencia temporal de los viajes

Se buscaron viajes cuya llegada fuera anterior o igual a la salida.

```sql
SELECT COUNT(*) AS viajes_con_fechas_invalidas
FROM trips
WHERE arrival_datetime <= departure_datetime;
```

**Resultado obtenido en DBeaver:** 0 viajes con llegada anterior o igual a la salida.

Esta consulta evalúa los viajes con fecha de llegada registrada. Los valores NULL de arrival_datetime no se incluyen en la comparación.

#### Horario de las entregas realizadas

Se buscaron entregas realizadas antes de la salida o después de la llegada del viaje asociado.

```sql
SELECT COUNT(*) AS entregas_fuera_del_viaje
FROM deliveries d
JOIN trips t ON d.trip_id = t.trip_id
WHERE d.delivered_datetime < t.departure_datetime
   OR d.delivered_datetime > t.arrival_datetime;
```

**Resultado obtenido en DBeaver:** 0 entregas fuera del intervalo de su viaje.

Las comparaciones con fechas NULL no permiten comprobar ambos límites. La existencia del viaje asociado se revisa mediante las validaciones de integridad referencial.

#### Peso de las entregas por viaje

Se buscaron viajes cuya suma del peso de los paquetes superara el peso total registrado para ese viaje.

```sql
SELECT COUNT(*) AS viajes_con_exceso_de_peso
FROM (
    SELECT t.trip_id
    FROM trips t
    JOIN deliveries d ON d.trip_id = t.trip_id
    GROUP BY t.trip_id, t.total_weight_kg
    HAVING SUM(d.package_weight_kg) > t.total_weight_kg
) AS revision;
```

**Resultado obtenido en DBeaver:** 0 viajes con exceso de peso de paquetes respecto al peso registrado del viaje.

#### Integridad referencial de los mantenimientos

Se buscaron mantenimientos sin un vehículo existente.

```sql
SELECT COUNT(*) AS mantenimientos_sin_vehiculo
FROM maintenance m
LEFT JOIN vehicles v ON m.vehicle_id = v.vehicle_id
WHERE v.vehicle_id IS NULL;
```

**Resultado obtenido en DBeaver:** 0 mantenimientos sin vehículo válido.